# LightGBM Model

In [ ]:
# Install LightGBM into this notebook's active kernel once if needed.
import importlib.util
import site
import subprocess
import sys

import lightgbm as lgb

import duckdb
import pandas as pd
from pathlib import Path
import os

In [3]:
# load the features from data/features

def load_features(file_name: str) -> pd.DataFrame:
    
    features_file = Path(
        "/accounts/masters/gautierep/footy_prediction/footy-prediction"
    ) / "data" / "features" / f"{file_name}.csv"

    features_df = pd.read_csv(features_file)

     # Turns catergorical columns into categorical veriables
    categorical_columns = ["HomeTeam", "AwayTeam"]
    for column in categorical_columns:
        categories = sorted(features_df[column].dropna().unique())
        features_df[column] = pd.Categorical(features_df[column], categories=categories)

    return features_df

features_df = load_features("features_data_1927")

In [12]:
features_df.shape #2690
(features_df['match_id']).max()
for i in range(4):
 print(i)
a = [1,2,3,4]
a.iloc[:2]

0
1
2
3


AttributeError: 'list' object has no attribute 'iloc'

In [14]:
import random
import itertools
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import log_loss

DEFAULT_FEATURES = [
    "home_points_last_5",
    "away_points_last_5",
    "HomeTeam",
    "AwayTeam",
    "home_points_home_last_5",
    "away_points_away_last_5",
    "home_goals_for_home_last_5",
    "away_goals_for_away_last_5",
    "market_home_prob_fair",
    "market_draw_prob_fair",
    "market_away_prob_fair",
    "goals_difference_last_5",
    "points_difference_last_5",
    "home_rest_days",
    "away_rest_days",
]

def K_fold_training(
    data: pd.DataFrame,
    feature_columns: list[str],
    target_column: str = "FTR",
    order_column: str = "match_id",
    test_fraction: float = 0.2,
    n_iter: int = 40,
    random_state: int = 42,
) -> pd.DataFrame:
    """Performs expanding walk-forward CV over a hyperparameter search space."""
    if not 0 < test_fraction < 1:
        raise ValueError("test_fraction must be between 0 and 1")

    # Column validation
    required_columns = [order_column, target_column, *feature_columns]
    missing_columns = sorted(set(required_columns) - set(data.columns))
    if missing_columns:
        raise KeyError(f"Missing columns: {missing_columns}")

    if len(feature_columns) != len(set(feature_columns)):
        raise ValueError("feature_columns contains duplicate names")

    # Chronological sort and drop missing rows
    ordered_data = data.sort_values(order_column).reset_index(drop=True).copy()
    ordered_data = ordered_data.dropna(subset=[target_column, *feature_columns])

    # Map target strings (H/D/A) to integers (0, 1, 2)
    target_map = {'H': 0, 'D': 1, 'A': 2}
    ordered_data['target'] = ordered_data[target_column].map(target_map)

    # 80/20 train-test temporal split
    split_index = int(len(ordered_data) * (1 - test_fraction))
    if split_index == 0 or split_index == len(ordered_data):
        raise ValueError("The time split leaves an empty train or test set")

    train_data = ordered_data.iloc[:split_index].sort_values(order_column).reset_index(drop=True)

    X_train = train_data[feature_columns].copy()
    y_train = train_data['target'].values

    # Calculate index cutoffs for 40%, 50%, 60%, 70%, and 80% overall dataset size
    n_train = len(X_train)
    per_40 = int(n_train * 0.50)      # 40% of total data = 50% of 80% train set
    per_50 = int(n_train * 0.625)     # 50% of total data = 62.5% of 80% train set
    per_60 = int(n_train * 0.75)      # 60% of total data = 75% of 80% train set
    per_70 = int(n_train * 0.875)     # 70% of total data = 87.5% of 80% train set
    per_80 = n_train                  # 80% of total data = 100% of 80% train set

    percentiles = [per_40, per_50, per_60, per_70, per_80]

    # Pragmatic, constrained search space for ~2,700 samples
    search_space = {
        "num_leaves": [7, 15, 31],
        "max_depth": [3, 4, 6],
        "learning_rate": [0.01, 0.03, 0.05],
        "min_child_samples": [20, 30, 50],
        "subsample": [0.7, 0.8, 1.0],
        "colsample_bytree": [0.7, 0.8, 1.0],
        "reg_lambda": [0.0, 1.0, 5.0],
    }

    # Generate all combinations and randomly sample `n_iter` configurations
    keys, values = zip(*search_space.items())
    all_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
    
    random.seed(random_state)
    param_samples = random.sample(all_combinations, min(n_iter, len(all_combinations)))

    results = []

    # Evaluate each hyperparameter configuration across the 4 walk-forward folds
    for param_idx, params in enumerate(param_samples):
        fold_log_losses = []
        fold_brier_scores = []

        for i in range(4):
            # Expanding train set up to percentile[i]; test set is the next 10% slice
            train_sub_X = X_train.iloc[:percentiles[i]]
            train_sub_y = y_train[:percentiles[i]]
            
            val_sub_X = X_train.iloc[percentiles[i]:percentiles[i+1]]
            val_sub_y = y_train[percentiles[i]:percentiles[i+1]]

            # Fit LightGBM model
            model = lgb.LGBMClassifier(
                objective="multiclass",
                num_class=3,
                n_estimators=150,
                random_state=random_state,
                verbosity=-1,
                **params
            )
            model.fit(train_sub_X, train_sub_y)

            # Predict probabilities
            val_preds = model.predict_proba(val_sub_X)

            # Compute Log Loss
            loss = log_loss(val_sub_y, val_preds)

            # Compute Multi-class Brier Score: Mean sum of squared errors
            val_sub_y_onehot = np.eye(3)[val_sub_y]
            brier = np.mean(np.sum((val_preds - val_sub_y_onehot) ** 2, axis=1))

            fold_log_losses.append(loss)
            fold_brier_scores.append(brier)

        # Record mean metrics across all 4 walk-forward folds
        result_entry = {
            "param_id": param_idx,
            "mean_log_loss": np.mean(fold_log_losses),
            "mean_brier_score": np.mean(fold_brier_scores),
            **params
        }
        results.append(result_entry)

    # Return results formatted as a sorted DataFrame
    return pd.DataFrame(results).sort_values("mean_log_loss").reset_index(drop=True)

K_fold_training(
    data=features_df,
    feature_columns=DEFAULT_FEATURES,
    target_column="FTR",
    order_column="match_id",
    test_fraction=0.2,
    random_state=42
)

,param_id,mean_log_loss,mean_brier_score,num_leaves,max_depth,learning_rate,min_child_samples,subsample,colsample_bytree,reg_lambda
0,8,0.972659,0.575052,31,4,0.01,30,0.7,0.7,0.0
1,23,0.973070,0.575096,31,4,0.01,30,0.7,0.8,0.0
2,3,0.973674,0.575601,15,4,0.01,30,0.7,0.8,1.0
3,38,0.975392,0.577133,7,6,0.01,20,1.0,1.0,1.0
4,33,0.975806,0.577458,31,3,0.01,20,0.8,0.8,0.0
5,21,0.976763,0.578166,7,3,0.01,20,1.0,1.0,5.0
6,17,0.976996,0.577789,31,4,0.01,20,0.8,1.0,5.0
7,7,0.979925,0.579733,7,4,0.03,30,0.7,0.8,5.0
8,11,0.980612,0.580774,7,4,0.03,50,0.7,0.8,5.0
9,15,0.980743,0.580380,7,3,0.03,30,0.7,0.7,0.0


In [15]:
# These are candidate pre-match features. The target FTR is kept separate.
# HS and AS are excluded because they are only known after the match.
DEFAULT_FEATURES = [
    "home_points_last_5",
    "away_points_last_5",
    "home_points_last_10",
    "away_points_last_10",
    "HomeTeam",
    "AwayTeam",
    "home_points_home_last_5",
    "away_points_away_last_5",
    "home_goals_for_home_last_5",
    "away_goals_for_away_last_5",
    "home_points_home_last_10",
    "away_points_away_last_10",
    "home_goals_for_home_last_10",
    "away_goals_for_away_last_10",
    "market_home_prob_fair",
    "market_draw_prob_fair",
    "market_away_prob_fair",
    "goals_difference_last_5",
    "points_difference_last_5",
    "home_shots_on_target_for_last_5",
    "away_shots_on_target_for_last_5",
    "home_fouls_for_last_5",
    "away_fouls_for_last_5",
    "home_corners_for_last_5",
    "away_corners_for_last_5",
    "home_rest_days",
    "away_rest_days",
]


FEATURE_GROUPINGS = {
    # 1. Benchmark: Pure market consensus (fair odds probabilities)
    "market_only": [
        "market_home_prob_fair",
        "market_draw_prob_fair",
        "market_away_prob_fair",
    ],

    # 2. Form & Standings: Points and simple differences (no market data)
    "form_only": [
        "home_points_last_5",
        "away_points_last_5",
        "home_points_last_10",
        "away_points_last_10",
        "points_difference_last_5",
        "goals_difference_last_5",
    ],

    # 3. Venue-Specific Form: Home performance at home, away performance away
    "venue_form_only": [
        "HomeTeam",
        "AwayTeam",
        "home_points_home_last_5",
        "away_points_away_last_5",
        "home_goals_for_home_last_5",
        "away_goals_for_away_last_5",
        "home_points_home_last_10",
        "away_points_away_last_10",
        "home_goals_for_home_last_10",
        "away_goals_for_away_last_10",
    ],

    # 4. Underlying Performance Metrics: Shots, fouls, corners, schedule rest
    "match_underlyings_only": [
        "home_shots_on_target_for_last_5",
        "away_shots_on_target_for_last_5",
        "home_fouls_for_last_5",
        "away_fouls_for_last_5",
        "home_corners_for_last_5",
        "away_corners_for_last_5",
        "home_rest_days",
        "away_rest_days",
    ],

    # 5. Complete Football Features: All physical/stats features (excluding market odds)
    "all_football_features": [
        "HomeTeam",
        "AwayTeam",
        "home_points_last_5",
        "away_points_last_5",
        "home_points_last_10",
        "away_points_last_10",
        "home_points_home_last_5",
        "away_points_away_last_5",
        "home_goals_for_home_last_5",
        "away_goals_for_away_last_5",
        "home_points_home_last_10",
        "away_points_away_last_10",
        "home_goals_for_home_last_10",
        "away_goals_for_away_last_10",
        "goals_difference_last_5",
        "points_difference_last_5",
        "home_shots_on_target_for_last_5",
        "away_shots_on_target_for_last_5",
        "home_fouls_for_last_5",
        "away_fouls_for_last_5",
        "home_corners_for_last_5",
        "away_corners_for_last_5",
        "home_rest_days",
        "away_rest_days",
    ],

    # 6. Market + Core Form: Market anchor enhanced by recent form & difference
    "market_plus_form": [
        "market_home_prob_fair",
        "market_draw_prob_fair",
        "market_away_prob_fair",
        "home_points_last_5",
        "away_points_last_5",
        "points_difference_last_5",
        "goals_difference_last_5",
    ],

    # 7. Combined Full Model: The entire feature space
    "full_combined": DEFAULT_FEATURES,
}

In [27]:
FEATURE_GROUPINGS['market_only']

['market_home_prob_fair', 'market_draw_prob_fair', 'market_away_prob_fair']

In [21]:
def run_grouping_cross_validation(features_df, feature_groupings):
    grouping_results = []

    for group_name, feature_list in feature_groupings.items():
        print(f"Running cross-validation for grouping: {group_name}...")
        
        cv_summary = K_fold_training(
            data=features_df,
            feature_columns=feature_list,
            target_column="FTR",
            order_column="match_id",
            n_iter=20  # Fast evaluation per set
        )
        
        # Extract top hyperparameter row (already sorted by mean_log_loss)
        best_row = cv_summary.iloc[0].to_dict()
        
        # Pull out metric columns
        log_loss_val = best_row.pop("mean_log_loss")
        brier_val = best_row.pop("mean_brier_score")
        param_id = best_row.pop("param_id")
        
        # The remaining key-value pairs in best_row are your hyperparameters
        grouping_results.append({
            "grouping": group_name,
            "feature_count": len(feature_list),
            "best_mean_log_loss": log_loss_val,
            "best_mean_brier_score": brier_val,
            "best_params": best_row  # Stores hyperparameters as a dictionary
        })

    # Format into a summary comparison DataFrame
    comparison_df = pd.DataFrame(grouping_results).sort_values("best_mean_log_loss").reset_index(drop=True)
    return comparison_df

best_params_groupings = run_grouping_cross_validation(features_df, FEATURE_GROUPINGS)
best_params_groupings

Running cross-validation for grouping: market_only...
Running cross-validation for grouping: form_only...
Running cross-validation for grouping: venue_form_only...
Running cross-validation for grouping: match_underlyings_only...
Running cross-validation for grouping: all_football_features...
Running cross-validation for grouping: market_plus_form...
Running cross-validation for grouping: full_combined...


,grouping,feature_count,best_mean_log_loss,best_mean_brier_score,best_params
0,market_only,3,0.964761,0.571793,"{'num_leaves': 15.0, 'max_depth': 3.0, 'learni..."
1,market_plus_form,7,0.969058,0.573897,"{'num_leaves': 7.0, 'max_depth': 4.0, 'learnin..."
2,full_combined,27,0.979212,0.579219,"{'num_leaves': 31.0, 'max_depth': 4.0, 'learni..."
3,all_football_features,24,1.005152,0.598707,"{'num_leaves': 31.0, 'max_depth': 4.0, 'learni..."
4,venue_form_only,10,1.010048,0.601983,"{'num_leaves': 31.0, 'max_depth': 4.0, 'learni..."
5,form_only,6,1.016855,0.607863,"{'num_leaves': 7.0, 'max_depth': 4.0, 'learnin..."
6,match_underlyings_only,8,1.024338,0.614851,"{'num_leaves': 7.0, 'max_depth': 3.0, 'learnin..."


In [28]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import log_loss

def evaluate_holdout_model(
    data: pd.DataFrame,
    feature_columns: list[str],
    hyperparams: dict,
    split_fraction: float = 0.8,
    target_column: str = "FTR",
    order_column: str = "match_id",
    random_state: int = 42
) -> dict:
    """Trains a LightGBM model on the initial split_fraction of the data (e.g. 0-80%)
    and evaluates Log Loss and Brier Score on the holdout slice (80-100%).
    """
    if not 0 < split_fraction < 1:
        raise ValueError("split_fraction must be between 0 and 1")

    # 1. Clean hyperparameter types (cast floats like 15.0 or 3.0 to int where required)
    cleaned_params = hyperparams.copy()
    int_keys = ["num_leaves", "max_depth", "min_child_samples"]
    for key in int_keys:
        if key in cleaned_params and cleaned_params[key] is not None:
            cleaned_params[key] = int(cleaned_params[key])

    # Remove non-LightGBM keys if present
    cleaned_params.pop("param_id", None)

    # 2. Chronological sort and drop missing targets/features
    required_cols = [order_column, target_column, *feature_columns]
    ordered_data = data.sort_values(order_column).dropna(subset=required_cols).reset_index(drop=True)

    # Map target strings (H=0, D=1, A=2)
    target_map = {'H': 0, 'D': 1, 'A': 2}
    ordered_data['target'] = ordered_data[target_column].map(target_map)

    # 3. Temporal Train / Holdout Split
    split_idx = int(len(ordered_data) * split_fraction)
    
    train_df = ordered_data.iloc[:split_idx]
    test_df = ordered_data.iloc[split_idx:]

    X_train = train_df[feature_columns]
    y_train = train_df['target'].values

    X_test = test_df[feature_columns]
    y_test = test_df['target'].values

    # 4. Train LightGBM Model
    model = lgb.LGBMClassifier(
        objective="multiclass",
        num_class=3,
        n_estimators=150,
        random_state=random_state,
        verbosity=-1,
        **cleaned_params
    )
    model.fit(X_train, y_train)

    # 5. Predict probabilities on the holdout set (80th - 100th percentile)
    test_preds = model.predict_proba(X_test)

    # 6. Compute Log Loss
    holdout_log_loss = log_loss(y_test, test_preds)

    # 7. Compute Multi-class Brier Score
    y_test_onehot = np.eye(3)[y_test]
    holdout_brier_score = np.mean(np.sum((test_preds - y_test_onehot) ** 2, axis=1))

    return {
        "train_matches": len(X_train),
        "test_matches": len(X_test),
        "split_fraction": split_fraction,
        "holdout_log_loss": round(holdout_log_loss, 4),
        "holdout_brier_score": round(holdout_brier_score, 4),
        "model": model
    }

params = best_params_groupings.iloc[0]['best_params']
evaluate_holdout_model(features_df, FEATURE_GROUPINGS['market_only'], hyperparams=params)

{'train_matches': 2152,
 'test_matches': 538,
 'split_fraction': 0.8,
 'holdout_log_loss': 1.0148,
 'holdout_brier_score': np.float64(0.6075),
 'model': LGBMClassifier(colsample_bytree=0.8, learning_rate=0.03, max_depth=3,
                n_estimators=150, num_class=3, num_leaves=15,
                objective='multiclass', random_state=42, reg_lambda=1.0,
                subsample=0.7, verbosity=-1)}

In [19]:
import numpy as np
import pandas as pd
import sklearn.metrics as metrics

def evaluate_market_baseline(
    data: pd.DataFrame,
    start_fraction: float = 0.8,
    end_fraction: float = 1.0,
    target_column: str = "FTR",
    order_column: str = "match_id",
    market_prob_cols: list[str] = [
        "market_home_prob_fair",
        "market_draw_prob_fair",
        "market_away_prob_fair"
    ]
) -> dict[str, float]:
    """Evaluates market-implied fair probabilities on a specific temporal slice of the dataset.
    
    Args:
        data: DataFrame containing target and market fair probabilities.
        start_fraction: Starting percentile fraction (e.g., 0.8 for 80th percentile).
        end_fraction: Ending percentile fraction (e.g., 1.0 for 100th percentile).
        target_column: Column name for target ('FTR').
        order_column: Column name for chronological ordering ('match_id').
        market_prob_cols: List of 3 probability columns [Home, Draw, Away].
        
    Returns:
        Dictionary containing Log Loss, Brier Score, and match count for the slice.
    """
    if not (0 <= start_fraction < end_fraction <= 1.0):
        raise ValueError("Must satisfy 0 <= start_fraction < end_fraction <= 1.0")

    # 1. Chronological order & drop missing values in relevant columns
    cols_to_check = [order_column, target_column, *market_prob_cols]
    ordered_data = data.sort_values(order_column).dropna(subset=cols_to_check).reset_index(drop=True)

    # 2. Slice temporal subset (e.g., 80% to 100%)
    start_idx = int(len(ordered_data) * start_fraction)
    end_idx = int(len(ordered_data) * end_fraction)
    
    test_slice = ordered_data.iloc[start_idx:end_idx].copy()
    
    if len(test_slice) == 0:
        raise ValueError(f"Slice range [{start_fraction}, {end_fraction}] resulted in zero matches.")

    # 3. Map targets (H=0, D=1, A=2)
    target_map = {'H': 0, 'D': 1, 'A': 2}
    y_true = test_slice[target_column].map(target_map).values

    # 4. Extract Market Probabilities array (N x 3)
    y_market_probs = test_slice[market_prob_cols].values

    # Normalize across rows to guarantee probabilities sum strictly to 1.0
    y_market_probs = y_market_probs / y_market_probs.sum(axis=1, keepdims=True)

    # 5. Compute Log Loss
    market_log_loss = metrics.log_loss(y_true, y_market_probs)

    # 6. Compute Multi-class Brier Score
    # Convert integer targets into one-hot binary matrix
    y_true_onehot = np.eye(3)[y_true]
    market_brier_score = np.mean(np.sum((y_market_probs - y_true_onehot) ** 2, axis=1))

    return {
        "match_count": len(test_slice),
        "start_percentile": f"{int(start_fraction * 100)}%",
        "end_percentile": f"{int(end_fraction * 100)}%",
        "market_log_loss": round(market_log_loss, 4),
        "market_brier_score": round(market_brier_score, 4)
    }

market_benchmark = evaluate_market_baseline(
    data=features_df,
    start_fraction=0.8,
    end_fraction=1.0,
    target_column="FTR",
    order_column="match_id"
)

print(market_benchmark)

{'match_count': 538, 'start_percentile': '80%', 'end_percentile': '100%', 'market_log_loss': 0.9992, 'market_brier_score': np.float64(0.5983)}


# APPENDIX